# EEG Data Processing
## Author: Arbintoro Mas (+62 877 1300 0050)
### For OpenBCI txt raw files and iMotions xlsx

In [1]:
import os
import csv
import time
import math
import statistics
import random
import pickle
import shutil
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure

import scipy
from scipy import signal, stats
from scipy.signal import filtfilt, butter, lfilter, sosfilt
from scipy.stats import entropy
from scipy.integrate import simpson

from sklearn import svm
from sklearn.decomposition import FastICA, PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

# !pip install pyeeg
# import pyeeg
# from pyegg import hjorth
from antropy import hjorth_params

import decimal
from decimal import *
getcontext().prec = 2

import seaborn as sns

import brainflow
from brainflow.board_shim import BoardShim, BrainFlowInputParams, LogLevels, BoardIds
from brainflow.data_filter import DataFilter, FilterTypes, AggOperations

import warnings
warnings.filterwarnings("ignore")

In [2]:
debug = False
### Start Form Input ###
directory_raw = "imotion_raw" #folder lokasi data raw, semua data raw kumpulkan di 1 folder ini tanpa di dalem sub folder lain
directory_ica = "imotion_ica"
directory_psd = "imotion_psd"
directory_graph = "imotion_graph"
directory_encoded = "imotion_encoded"
directory_mav = "imotion_mav"
directory_compare = "imotion_compare"
directory_analysis = "imotion_analysis"
directory_unique = "imotion_unique"
directory_empty = "imotion_raw_empty"
directory_corrupt = "imotion_raw_corrupt"

channels = ["F8","F7","T8","T7","P8","P7","O2","O1"] #channel yang digunakan untuk rekaman, isi harus urut dan gunakan 'NaN' jika channel tidak digunakan
channeling = ["F8","F7","T8","T7","P8","P7","O2","O1"] #channel yang dipilih untuk analisa encode
brainwaves = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
subbanding = ["Delta", "Theta", "Alpha", "Beta", "Gamma"] #subband yang dipilih untuk analisa encode
label_prefix = '_' #tanda baca sebelum pembeda label misal "arbi_senang", "arbi_sedih" atau "arbi-senang" berarti '-'
label_prefix_start = 1 # 0 = awal "_label", 1 = akhir "label_"
labels = {
    "baseline1": {"start": 0, "seconds": 100},
    "hs": {"start": 10, "seconds": 100},
    "bengbeng": {"start": 10, "seconds": 30},
    "marjan": {"start": 10, "seconds": 30},
    "ns_gandaria": {"start": 10, "seconds": 30},
    "ns_jeruk": {"start": 10, "seconds": 30},
    "pucuk": {"start": 10, "seconds": 30},
    "shopee": {"start": 10, "seconds": 30},
    "lokalate": {"start": 10, "seconds": 30},
    "ts_nana": {"start": 10, "seconds": 30},
    "ts_pat": {"start": 10, "seconds": 30},
    "honda": {"start": 10, "seconds": 90},
    #pengecoh
    "abc": {"start": 10, "seconds": 30},
    "bearbrand": {"start": 10, "seconds": 15},
    "hilo_platinum": {"start": 10, "seconds": 30},
    "hilo_many": {"start": 10, "seconds": 30},
    "bukalapak": {"start": 10, "seconds": 90},
    "dancow": {"start": 10, "seconds": 42},
    "ensure": {"start": 10, "seconds": 30},
    "gerry": {"start": 10, "seconds": 30},
    "indomie": {"start": 10, "seconds": 29},
    "kopikenangan": {"start": 10, "seconds": 60},
    "marimas": {"start": 10, "seconds": 33},
    "mmpo": {"start": 10, "seconds": 30},
    "nescafe": {"start": 10, "seconds": 90},
    "oreo": {"start": 10, "seconds": 60},
    "samsung": {"start": 10, "seconds": 30},
    "smartfren": {"start": 10, "seconds": 41},
    "tokped": {"start": 10, "seconds": 15},
    "torabika": {"start": 10, "seconds": 30},
    "tehbotol": {"start": 10, "seconds": 30}
}
# labels = {
#     "bengbeng": {"start": 10, "seconds": 5}
# }

ext_raw = ".xlsx"

do_psd = True #jadikan True jika mau jalanin proses PSD
do_analyze_psd = False #jadikan True jika mau jalanin analisa PSD juga
do_analyze_encode = False #jadikan True jika mau jalanin analisa encode juga
train_encode = 80 #berapa persentase data encode yang digunakan untuk train (train test split)

live_filename = "Live_EEG" #default filename

timer = 5 #berapa detik perekaman live EEG direkam

second_start = 0 #berapa detik pertama yang dipotong dari data mentah untuk diproses
seconds = 5 #berapa detik data yang akan diproses

measurement_time = 5 #detik untuk live graph recording (coming soon)
interval_sec = 0.1 #interval untuk live graph recording (coming soon)

sample_frequency = 250 #sample frequency default OpenBCI = 250
sample_rate = 256 #sampling data per detik OpenBCI = 256
notch_freq = 50.0 #notch frequency
quality_factor = 2.0
lowcut = 0.1
highcut = 49
order = 5
win = 1024
chunks = 0.25 #detik chunk, harus kelipatan 2 (0.125 = 1/8)

treshold_ica = True #potong ICA jika data ICA melewati atau kurang dari batas treshold
high_ica = 100
low_ica = -100

iter_seconds = 4 #hiraukan

print_debug = False
print_raw = True
print_filter = False
print_butter = False
print_ica = False
print_psd = True
### End Form Input ####

final_graph = True

if not os.path.exists(directory_ica):
    os.makedirs(directory_ica)
if not os.path.exists(directory_psd):
    os.makedirs(directory_psd)
if not os.path.exists(directory_graph):
    os.makedirs(directory_graph)
if not os.path.exists(directory_raw):
    os.makedirs(directory_raw)
if not os.path.exists(directory_mav):
    os.makedirs(directory_mav)
if not os.path.exists(directory_compare):
    os.makedirs(directory_compare)
if not os.path.exists(directory_encoded):
    os.makedirs(directory_encoded)
if not os.path.exists(directory_analysis):
    os.makedirs(directory_analysis)
if not os.path.exists(directory_unique):
    os.makedirs(directory_unique)
if not os.path.exists(directory_corrupt):
    os.makedirs(directory_corrupt)
if not os.path.exists(directory_empty):
    os.makedirs(directory_empty)
    
start_time = time.time()
end_time = time.time()

cut = list(range(1, len(channels)+1))
# print(cut)
dict_zip = dict(zip(cut, channels))
# print(dict_zip)

frame_start = second_start * sample_rate
frame_end = ((second_start + seconds) * sample_rate) - 1

nyq = 0.5 * sample_frequency
low = lowcut / nyq
high = highcut / nyq

sub_freqs = {}
sub_freqs["Delta"] = {"Low":0.1, "High":4}
sub_freqs["Theta"] = {"Low":4, "High":8}
sub_freqs["Alpha"] = {"Low":8, "High":13}
sub_freqs["Beta"] = {"Low":13, "High":35}
sub_freqs["Gamma"] = {"Low":35, "High":50}
low_d, high_d = 0.1, 4 # Delta
low_t, high_t = 4, 8 # Theta
low_a, high_a = 8, 13 # Alpha
low_b, high_b = 13, 35 # Beta
low_g, high_g = 35, 50 # Gamma

b_notch, a_notch = signal.iirnotch(notch_freq, quality_factor, sample_frequency)
b_butter,a_butter=scipy.signal.butter(order,[low, high], 'bandpass', analog=False)

b_d, a_d = scipy.signal.butter(order, [low_d / nyq, high_d / nyq], 'bandpass', analog=False)
b_t, a_t = scipy.signal.butter(order, [low_t / nyq, high_t / nyq], 'bandpass', analog=False)
b_a, a_a = scipy.signal.butter(order, [low_a / nyq, high_a / nyq], 'bandpass', analog=False)
b_b, a_b = scipy.signal.butter(order, [low_b / nyq, high_b / nyq], 'bandpass', analog=False)
b_g, a_g = scipy.signal.butter(order, [low_g / nyq, high_g / nyq], 'bandpass', analog=False)

ICA = FastICA(n_components=1)

os_raw = os.fsencode(directory_raw)
os_ica = os.fsencode(directory_ica)
os_processed = os.fsencode(directory_psd)
os_analysis = os.fsencode(directory_analysis)
os_mav = os.fsencode(directory_mav)
os_compare = os.fsencode(directory_compare)
os_encoded = os.fsencode(directory_encoded)
os_corrupt = os.fsencode(directory_corrupt)
os_empty = os.fsencode(directory_empty)

In [3]:
def getLiveData():
    global timer
    global channels
    global dict_zip
    global directory_raw
    global live_filename
    
    BoardShim.enable_dev_board_logger()

    serial_port = "COM3"
    board_id = BoardIds.CYTON_BOARD.value
    
    params = BrainFlowInputParams()
    params.serial_port = serial_port
    
    board = BoardShim(board_id, params)
    
    board.prepare_session()
    board.start_stream()
    time.sleep(timer)
    data = board.get_board_data()
    board.stop_stream()
    board.release_session()
    
    df_live = pd.DataFrame(np.transpose(data))
    
    N = len(channels) + 1
    
    dfRaw = df_live.iloc[1: , 1:N].rename(columns = dict_zip, inplace = False)
    
    fn = directory_raw + "/" + live_filename + ".csv"

    DataFilter.write_file(data, fn, 'w')
    
    return dfRaw

def getChannel(idx):
    global dict_zip
    
    channel = dict_zip.get(idx+1)
    if channel is None:
        channel = str(idx+1)
    return channel

def pureFilename(filepath, idx = ""):
    global ext_raw
    
    name = filepath.replace(" ", "_")
    name = name.replace(ext_raw, "")
    if idx != "":
        name = name + "_" + getChannel(idx)
    return name

def chunky2(data, n, timestamp = ""):
    global sample_rate
    seconds = len(data) / sample_rate
    split = int(math.ceil(seconds / n))

    result = np.array_split(data, split)
    return np.array_split(data, split)

def chunky(data, n, timestamps):
    
    interval_ms = n * 1000
    chunks = []
    current_chunk = []
    current_time = timestamps.iloc[0]
    first = timestamps.index[0]
    
    for idx, timestamp in timestamps.items():
        if timestamp < current_time + interval_ms:
            current_chunk.append(float(data.iloc[idx - first]))
        else:
            chunks.append(pd.DataFrame({0:current_chunk.copy()}))
            current_chunk = [float(data.iloc[idx - first])]
            current_time = current_time + interval_ms

    if len(current_chunk) > 1:
        chunks.append(pd.DataFrame({0:current_chunk}))
        
    return chunks

def meanMulti(directory):
    means = {}
    for file in directory:
        for idx, data in enumerate(file):
            channel = getChannel(idx)
            mean = sum(data) / len(data)
            if channel in means:
                means[channel].append(mean)
            else:
                means[channel] = [mean]
        
    for key in means:
        mean = sum(means[key]) / len(means[key])
        if mean > 0.01:
            print(key + " = " + str(round(mean, 2)))
        else:
            print(key + " = " + str(round(mean, 10)))
            
def getSampleRate(timestamps):
    start = 0
    end = 0
    count = 0
    for timestamp in timestamps:
        count = count + 1
        if start == 0:
            start = timestamp
        
        dif = timestamp - start
        
        if dif >= 1000:
            return count
        
def getFrame(df):
    global seconds
    global second_start
    
    df.sort_values('Timestamp', ascending=True, inplace=True)
    
    begin = df['Timestamp'].iloc[0]
    start = begin + second_start * 1000
    duration = seconds * 1000
    max_end = start + duration
    
    filtered_df = df[(df['Timestamp'] >= start) & (df['Timestamp'] <= max_end)]
    end = filtered_df['Timestamp'].max()
    
    frame_start = (df[df['Timestamp'] <= start]['Timestamp']).idxmax()
    frame_end = (df[df['Timestamp'] <= end]['Timestamp']).idxmax()
    
    return frame_start, frame_end

def cleanRawData(df):
#     for index, row in df.iterrows():
#         if math.isnan(row['CH.0']):
#             df.drop(index, inplace=True)
    df = df[df['CH.0'].notna()]
    return df
        
def readRawData(filepath):
    global cut
    global dict_zip
    
    global ext_raw
    
    global print_debug
    global print_raw
    
    if print_debug:
        print()
        print("Read Raw Data")
        print()
    
    skiprows = 7
    sep = ", "
    if ext_raw == '.xlsx':
        df =  pd.read_excel(filepath)
        dfRaw = cleanRawData(df)
    else:
        if ext_raw == '.csv':
            skiprows = 1
            sep = "	"
        elif ext_raw == '.txt':
            skiprows = 7
            sep = ", "
        df =  pd.read_csv(filepath, skiprows = skiprows, header=None, sep=sep, engine="python")
#         print("\n\ndf")
#         print(df)
#         print("\n\n\n")
        dfRaw = df[cut]
        dfRaw = dfRaw.rename(columns = dict_zip, inplace = False)
        
#     dfRaw = df[cut]
#     dfRaw = dfRaw.rename(columns = dict_zip, inplace = False)
    
    if print_raw:
        print(dfRaw)
    
    return dfRaw

def plotRawData(dfRaw, filename):
    global print_debug
    
    if print_debug:
        print()
        print("Plot Raw Data")
        print()
    
    dfRaw.plot(kind='line',figsize=(15,6))
    plt.style.use('seaborn-colorblind')
    plt.style.use('seaborn-whitegrid')
    plt.title("Raw " + filename)
    plt.xlabel("Frame")
    plt.ylabel("Value")
    plt.legend()
    plt.show()
    
def filterRawData(dfRaw):
    global b_notch
    global a_notch
    
    global print_debug
    global print_filter
    
    if print_debug:
        print()
        print("Filter Raw")
        print()

    outputSignals = []
    
    for idx, raw in dfRaw.iteritems():
        filt = signal.filtfilt(b_notch, a_notch, raw)
        dfFilt = pd.DataFrame(filt)
        outputSignals.append(dfFilt)
        
        if print_filter:
            print(str(idx+1) + " = ")
            print(dfFilt)
            print()

          
#     print("Output Signals:\n\n")
#     print(outputSignals)
#     print("\nEND OS\n\n")
    
    #dictSemua = {'T3': outputSignals[0], 'T4': outputSignals[1], 'T5': outputSignals[2], 'T6': outputSignals[3], 'O1': outputSignals[4],'O2': outputSignals[5]} 
    
    #print("OutputSignals[0]")
    #print(outputSignals[0])
    #print("\nEND df OS\n")
    #print(dictSemua)
    
    #isiSemua = pd.DataFrame(dictSemua)
    
    #print(isiSemua)
                
    #plt.figure(2)
    #plt.style.use('seaborn-colorblind')
    #plt.style.use('seaborn-whitegrid')
    #plt.plot(outputSignals[0])
    #plt.plot(isiSemua)
    #plt.ylabel("Output Signals")
    #plt.show()
    
    return outputSignals

def plotFilterData(outputSignals, filename):
    global print_debug
    
    if print_debug:
        print()
        print("Plot Filter Data")
        print()
    
    data = []
    plt.figure(figsize=(15,6))
    for idx, os in enumerate(outputSignals):
        plt.plot(os, label = getChannel(idx))
#         data.append(os)
#     data.plot(figsize=(15, 6))
    plt.style.use('seaborn-colorblind')
    plt.style.use('seaborn-whitegrid')
    plt.title("Notch Filter " + filename)
    plt.xlabel("Frame")
    plt.ylabel("Value")
    plt.legend()
    plt.show()
            
def butterData(outputSignals):
    global b_butter
    global a_butter
    
    global print_debug
    global print_butter
    
    if print_debug:
        print()
        print("Butter Data")
        print()
    
    butterData = []
    
    for idx, outputSignal in enumerate(outputSignals):
        y = scipy.signal.filtfilt(b_butter, a_butter, outputSignal, axis=0)
        dfButter = pd.DataFrame(y)
        butterData.append(dfButter)
        
        if print_butter:
            print(str(idx+1) + " = ")
            print(dfButter)
            print()
          
#     print("Butter Data:\n\n")
#     print(butterData)
#     print("\nEND BD\n\n")
                    
    #plt.figure(3)
    #plt.style.use('seaborn-colorblind')
    #plt.style.use('seaborn-whitegrid')
    #plt.plot(outputSignals[0])
    #plt.plot(butterData)
    #plt.ylabel("Butter Data")
    #plt.show()
    
    return butterData

def plotButterData(butterData, filename):
    global print_debug
    
    if print_debug:
        print()
        print("Plot Butter Data")
        print()
    
    data = []
    plt.figure(figsize=(15,6))
    for idx, bd in enumerate(butterData):
        plt.plot(bd, label = getChannel(idx))
#         data.append(bd)
#     data.plot(figsize=(15, 6))
    plt.style.use('seaborn-colorblind')
    plt.style.use('seaborn-whitegrid')
    plt.title("Bandpass Filter " + filename)
    plt.xlabel("Frame")
    plt.ylabel("Value")
    plt.legend()
    plt.show()

def ICAData(butterData):
    global treshold_ica
    global high_ica
    global low_ica
    global ICA
    
    global print_debug
    global print_ica
    
    if print_debug:
        print()
        print("ICA Data")
        print()
    
    ICAData = []
    
    for idx, butter in enumerate(butterData):
        InpData = pd.DataFrame(data=butter)
        X = InpData.values
        IndependentComponentValues = ICA.fit_transform(X)
        ReducedData = pd.DataFrame(data = IndependentComponentValues)
        ReducedData = ReducedData * 1000
        if treshold_ica:
            for RD in ReducedData:
                if int(RD) > int(high_ica):
                    RD = high_ica
                elif int(RD) < int(low_ica):
                    RD = low_ica
        ReducedData = ReducedData.round(3)
        ICAData.append(ReducedData)
        
        if print_ica:
            print(str(idx+1) + " = ")
            print(ReducedData)
            print()
                      
#     print("ICA Data:\n\n")
#     print(ICAData)
#     print("\nEND ID\n\n")

#     print("ICAData")
#     print(ICAData)
    return ICAData
        
def plotAmp(ICAData, filename, print_singles):
    global print_debug
    
    if print_debug:
        print()
        print("Plot Amplitude")
        print()

    plt.figure(figsize=(15,6))
    for idx, icad in enumerate(ICAData):
        plt.plot(icad, label = getChannel(idx))
    plt.style.use('seaborn-colorblind')
    plt.style.use('seaborn-whitegrid')
    plt.title("Amplitude " + filename)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (uV)")
    plt.legend()
    plt.show()
    
    if print_singles:
        for idx, icad in enumerate(ICAData):
            plt.figure(figsize=(15,6))
            plt.style.use('seaborn-colorblind')
            plt.style.use('seaborn-whitegrid')
            plt.plot(icad, label = getChannel(idx))
            plt.title("Amplitude " + filename)
            plt.xlabel("Time (s)")
            plt.ylabel("Amplitude (uV)")
            plt.legend()
            plt.show()
        
def saveICAcsv(ICAData, filename):
    global print_debug
    
    if print_debug:
        print()
        print("Save ICA csv")
        print()
    
    for idx, icad in enumerate(ICAData):
        name = pureFilename(filename, idx)
            
        df_ica = pd.DataFrame(icad)
        ica_file = directory_ica + "/" + name + ".csv"
        fileOverwrite(ica_file)
        df_ica.to_csv(ica_file, header = False, index = False)
        
# def butter_bandpass(lowcut, highcut, fs, order=5):
#     nyq = 0.5 * fs
#     low = lowcut / nyq
#     high = highcut / nyq
#     b, a = butter(order, [lowcut, highcut], btype='band')
#     return b, a


# def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
#     b, a = butter_bandpass(lowcut, highcut, fs, order=order)
#     y = lfilter(b, a, data[0])
#     return y
        
def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    sos = butter(order, [low, high], analog=False, btype='band', output='sos')
    return sos

def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    sos = butter_bandpass(lowcut, highcut, fs, order=order)
    y = sosfilt(sos, data[0])
    return y
        
def bandpassFilter(ICAData, filename):
    global low_d, high_d
    global low_t, high_t
    global low_a, high_a
    global low_b, high_b
    global low_g, high_g
    global b_d, a_d
    global b_t, a_t
    global b_a, a_a
    global b_b, a_b
    global b_g, a_g
    global sample_frequency
    global chunks
    global directory_ica
    
    delta = []
    theta = []
    alpha = []
    beta = []
    gamma = []
    all_list = []
    
    fs = sample_frequency
    
    pd.set_option('display.float_format', '{:.5f}'.format)
    
    for idx, icad in enumerate(ICAData):
#         y_d = scipy.signal.filtfilt(b_d, a_d, icad, axis=0)
        y_d = butter_bandpass_filter(icad, low_d, high_d, fs)
#         y_d = y_d * 1000
        df_d = pd.DataFrame(y_d)
        delta.append(df_d)

#         y_t = scipy.signal.filtfilt(b_t, a_t, icad, axis=0)
        y_t = butter_bandpass_filter(icad, low_t, high_t, fs)
#         y_t = y_t * 1000
        df_t = pd.DataFrame(y_t)
        theta.append(df_t)

#         y_a = scipy.signal.filtfilt(b_a, a_a, icad, axis=0)
        y_a = butter_bandpass_filter(icad, low_a, high_a, fs)
#         y_a = y_a * 1000
        df_a = pd.DataFrame(y_a)
        alpha.append(df_a)
        
#         y_b = scipy.signal.filtfilt(b_b, a_b, icad, axis=0)
        y_b = butter_bandpass_filter(icad, low_b, high_b, fs)
#         y_b = y_b * 1000
        df_b = pd.DataFrame(y_b)
        beta.append(df_b)

#         y_g = scipy.signal.filtfilt(b_g, a_g, icad, axis=0)
        y_g = butter_bandpass_filter(icad, low_g, high_g, fs)
#         y_g = y_g * 1000
        df_g = pd.DataFrame(y_g)
        gamma.append(df_g)
        
        cn = getChannel(idx) + "_"
        cn_ica = cn + "ICA"
        df_ica = pd.DataFrame({cn_ica:icad[0]})
        all_list.append(df_ica)
        cn_d = cn + "Delta"
#         y_d = [j for sub in y_d for j in sub]
        df_d = pd.DataFrame({cn_d:y_d})
        all_list.append(df_d)
        cn_t = cn + "Theta"
#         y_t = [j for sub in y_t for j in sub]
        df_t = pd.DataFrame({cn_t:y_t})
        all_list.append(df_t)
        cn_a = cn + "Alpha"
#         y_a = [j for sub in y_a for j in sub]
        df_a = pd.DataFrame({cn_a:y_a})
        all_list.append(df_a)
        cn_b = cn + "Beta"
#         y_b = [j for sub in y_b for j in sub]
        df_b = pd.DataFrame({cn_b:y_b})
        all_list.append(df_b)
        cn_g = cn + "Gamma"
#         y_g = [j for sub in y_g for j in sub]
        df_g = pd.DataFrame({cn_g:y_g})
        all_list.append(df_g)
        
    index_label = "Index"
    all_filename = directory_ica + "/" + pureFilename(filename) + ".xlsx"
    fileOverwrite(all_filename)
    xls_list = pd.concat(all_list, axis=1)
    xls_list.index += 1
    xls_list.to_excel(all_filename, index_label=index_label)
    
    return {"Delta":delta, "Theta":theta, "Alpha":alpha, "Beta":beta, "Gamma":gamma}
            
def mavData(ICAData, filename, print_mav, timestamp):
    global chunks
    global directory_mav
    global directory_psd
    global sub_freqs
    global do_psd
    
    global print_debug
    
    global debug
    
    if print_debug:
        print()
        print("MAV")
        print()
        
    bandpassICAData = bandpassFilter(ICAData, filename)
    
    chunk_count = True
    chunk_temp = []

    mav_delta_list = []
    mav_theta_list = []
    mav_alpha_list = []
    mav_beta_list = []
    mav_gamma_list = []

    mav_delta = {}
    mav_theta = {}
    mav_alpha = {}
    mav_beta = {}
    mav_gamma = {}
    
    mav_list = []
    
    for bw, data in bandpassICAData.items():
        for idx, icad in enumerate(data):
            name = pureFilename(filename, idx)
            data_chunks = chunky(icad, chunks, timestamp)
            channel = getChannel(idx)

            if debug:
                print("\n------------------------------\n" + channel + "\n------------------------------\n")        
                print("Data Chunks = " + str(len(data_chunks)))
            counter_chunk = 1
            
#             psd_temp = []
            mean_temp = []
            mav_temp = []
            var_temp = []
            sd_temp = []
            en_temp = []
            mob_temp = []
            com_temp = []

            for chunk in data_chunks:
                if debug:
                    print()
                    print(str(counter_chunk) + ".")
#                 mean, mav, var, sd, en = calculateFeature(chunk)
                mean, mav, var, sd, en, mob, com = calculateFeature(chunk)

                if chunk_count:
                    chunk_temp.append(len(chunk))
                
#                 psd_temp.append(psd)
                mean_temp.append(mean)
                mav_temp.append(mav)
                var_temp.append(var)
                sd_temp.append(sd)
                en_temp.append(en)
                mob_temp.append(mob)
                com_temp.append(com)

                counter_chunk += 1
            
            if chunk_count:
                mav_list.append(pd.DataFrame(chunk_temp, columns=["Data Length"]))
                chunk_count = False
            cn = channel + "_" + bw + "_"
            temp_list = {}
            temp_list[cn + "Mean"] = mean_temp
            temp_list[cn + "Mean Absolute Value"] = mav_temp
            temp_list[cn + "Variance"] = var_temp
            temp_list[cn + "Standard Deviation"] = sd_temp
            temp_list[cn + "Entropy"] = en_temp
            temp_list[cn + "Hjorth Mobility"] = mob_temp
            temp_list[cn + "Hjorth Complexity"] = com_temp

            mav_list.append(pd.DataFrame(temp_list))
            
            if bw == "Delta":
                mav_delta = {"Filename":name+"_Delta", "Mean":mean_temp, "MAV":mav_temp, "SD":sd_temp}
                mav_delta_list.append(mav_delta)
            elif bw == "Theta":
                mav_theta = {"Filename":name+"_Theta", "Mean":mean_temp, "MAV":mav_temp, "SD":sd_temp}
                mav_theta_list.append(mav_theta)
            elif bw == "Alpha":
                mav_alpha = {"Filename":name+"_Alpha", "Mean":mean_temp, "MAV":mav_temp, "SD":sd_temp}
                mav_alpha_list.append(mav_alpha)
            elif bw == "Beta":
                mav_beta = {"Filename":name+"_Beta", "Mean":mean_temp, "MAV":mav_temp, "SD":sd_temp}
                mav_beta_list.append(mav_beta)
            elif bw == "Gamma":
                mav_gamma = {"Filename":name+"_Gamma", "Mean":mean_temp, "MAV":mav_temp, "SD":sd_temp}
                mav_gamma_list.append(mav_gamma)
    
    index_label = "Chunk (" + str(chunks) + " seconds)"
    mav_filename = directory_mav + "/" + pureFilename(filename) + ".xlsx"
    fileOverwrite(mav_filename)
    all_list = pd.concat(mav_list, axis=1)
    all_list.index += 1
    all_list.to_excel(mav_filename, index_label=index_label)
        
    return mav_delta_list, mav_theta_list, mav_alpha_list, mav_beta_list, mav_gamma_list
        
def plotMAV(counter, data, title, name, print_mav):
    global channels
    
    plt.figure(counter, figsize=(15,6))
    plt.style.use('seaborn-colorblind')
    plt.style.use('seaborn-whitegrid')
    for idx, row in enumerate(data):
        plt.plot(row, label = getChannel(idx))
    plt.ylabel("SD " + title)
    plt.legend()
    for channel in channels:
        if channel != 'Nan':
            find = "_" + channel
            name = name.replace(find, "")
    graph_title = directory_graph + "/Graph_SD_" + name + "_" + title + ".png"
    fileOverwrite(graph_title)
    plt.savefig(graph_title)
    if print_mav:
        plt.show()
        for idx, row in enumerate(data):
            average = sum(row)/len(row)
            print(getChannel(idx) + " Average = " + "{:.2f}".format(average) + " uV^2")
    else:
        plt.close(counter)
        
def welchData(data):
    global sample_frequency
    global sample_rate
    global win
    global chunks
    
    global debug
    
    fs = 100
    nperseg = fs * chunks
    
#     freqs, psd = signal.periodogram(data[0], fs=100, scaling='density')
    freqs, psd = signal.welch(data[0], fs=fs, nperseg=nperseg)
#     freqs, psd = signal.welch(data[0], fs=100, nperseg=(len(data)*4))
    
    freq_res = freqs[1] - freqs[0]
                      
#     print("Len Data = " + str(len(data)))
#     print("freqs (" + str(len(freqs)) + "):")
#     print(freqs)
#     print("freq_res:")
#     print(freq_res)
#     print("psd (" + str(len(psd)) + "):")
#     print(psd)
#     print()
    return freqs, psd, freq_res
            
def psdData(ICAData, filename, print_psd, timestamp):
    global chunks
    global sub_freqs
    global brainwaves
    global low_d
    global low_t
    global low_a
    global low_b
    global low_g
    global high_d
    global high_t
    global high_a
    global high_b
    global high_g
    
    global print_debug
    
    global debug
    
    if print_debug:
        print()
        print("PSD")
        print()
    
    chunk_count = True
    chunk_temp = []
        
    delta = []
    theta = []
    alpha = []
    beta = []
    gamma = []
    
    psd_list = []
    all_list = []
    
    for idx, icad in enumerate(ICAData):
        name = pureFilename(filename, idx)
        data_chunks = chunky(icad, chunks, timestamp)
        channel = getChannel(idx)
        
        if debug:
            print("\n------------------------------\n" + channel + "\n------------------------------\n")        
            print("Data Chunks = " + str(len(data_chunks)))

        psd_delta = []
        psd_theta = []
        psd_alpha = []
        psd_beta = []
        psd_gamma = []
        temp_list = {}

        counter_chunk = 0
        for chunk in data_chunks:
            counter_chunk += 1
            if debug:
                print()
                print(str(counter_chunk) + ".")
                
#             if len(chunk) > 1:
            if chunk_count:
                chunk_temp.append(len(chunk))

            freqs, psd, freq_res = welchData(chunk)

            idx_band_d = np.logical_and(freqs >= low_d, freqs <= high_d)
            band_power_d = None
            if len(psd[idx_band_d]):
                if len(psd[idx_band_d]) > 1:
                    band_power_d = simpson(psd[idx_band_d], dx=freq_res)
                else:
                    band_power_d = psd[idx_band_d][0]
            psd_delta.append(band_power_d)

            idx_band_t = np.logical_and(freqs >= low_t, freqs < high_t)
            band_power_t = None
            if len(psd[idx_band_t]):
                if len(psd[idx_band_d]) > 1:
                    band_power_t = simpson(psd[idx_band_t], dx=freq_res)
                else:
                    band_power_t = psd[idx_band_t][0]
            psd_theta.append(band_power_t)

            idx_band_a = np.logical_and(freqs >= low_a, freqs < high_a)
            band_power_a = None
            if len(psd[idx_band_a]):
                if len(psd[idx_band_d]) > 1:
                    band_power_a = simpson(psd[idx_band_a], dx=freq_res)
                else:
                    band_power_a = psd[idx_band_a][0]
            psd_alpha.append(band_power_a)

            idx_band_b = np.logical_and(freqs >= low_b, freqs < high_b)
            band_power_b = None
            if len(psd[idx_band_b]):
                if len(psd[idx_band_d]) > 1:
                    band_power_b = simpson(psd[idx_band_b], dx=freq_res)
                else:
                    band_power_b = psd[idx_band_b][0]
            psd_beta.append(band_power_b)

            idx_band_g = np.logical_and(freqs >= low_g, freqs < high_g)
            band_power_g = None
            if len(psd[idx_band_g]):
                if len(psd[idx_band_d]) > 1:
                    band_power_g = simpson(psd[idx_band_g], dx=freq_res)
                else:
                    band_power_g = psd[idx_band_g][0]
            psd_gamma.append(band_power_g)
            
#             temp_list[str(counter_chunk)] = [band_power_d, band_power_t, band_power_a, band_power_b, band_power_g]
        
        if chunk_count:
            all_list.append(pd.DataFrame(chunk_temp, columns=["Data Length"]))
            chunk_count = False
            
        delta.append(psd_delta)
        theta.append(psd_theta)
        alpha.append(psd_alpha)
        beta.append(psd_beta)
        gamma.append(psd_gamma)
        
#         columns = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
#         psd_filename = directory_psd + "/" + name + ".xlsx"
#         fileOverwrite(psd_filename)
#         psd_list = pd.DataFrame.from_dict(temp_list, orient='index', columns=columns)
#         psd_list.to_excel(psd_filename, index_label='Chunks')
        
        cn = channel + "_Delta"
        all_list.append(pd.DataFrame({cn:psd_delta}))
        cn = channel + "_Theta"
        all_list.append(pd.DataFrame({cn:psd_theta}))
        cn = channel + "_Alpha"
        all_list.append(pd.DataFrame({cn:psd_alpha}))
        cn = channel + "_Beta"
        all_list.append(pd.DataFrame({cn:psd_beta}))
        cn = channel + "_Gamma"
        all_list.append(pd.DataFrame({cn:psd_gamma}))
        
    index_label = "Chunk (" + str(chunks) + " seconds)"
    all_filename = directory_psd + "/" + pureFilename(filename) + ".xlsx"
    fileOverwrite(all_filename)
    xls_list = pd.concat(all_list, axis=1)
    xls_list.index += 1
    xls_list.to_excel(all_filename, index_label=index_label)
                
#     if final_graph:
#         plotPSD(1, delta, "Delta", name, print_psd)
#         plotPSD(2, theta, "Theta", name, print_psd)
#         plotPSD(3, alpha, "Alpha", name, print_psd)
#         plotPSD(4, beta, "Beta", name, print_psd)
#         plotPSD(5, gamma, "Gamma", name, print_psd)
        
    return delta, theta, alpha, beta, gamma

def plotPSD(counter, data, title, name, print_psd):
    global channels
    
    plt.figure(counter, figsize=(15,6))
    plt.style.use('seaborn-colorblind')
    plt.style.use('seaborn-whitegrid')
    for idx, row in enumerate(data):
        plt.plot(row, label = getChannel(idx))
    plt.ylabel("PSD " + title)
    plt.legend()
    for channel in channels:
        if channel != 'Nan':
            find = "_" + channel
            name = name.replace(find, "")
    graph_title = directory_graph + "/Graph_PSD_" + name + "_" + title + ".png"
    fileOverwrite(graph_title)
    plt.savefig(graph_title)
    if print_psd:
        plt.show()
        for idx, row in enumerate(data):
            average = sum(row)/len(row)
            print(getChannel(idx) + " Average = " + "{:.2f}".format(average) + " uV^2")
    else:
        plt.close(counter)
        
def analyzeStatistic(analyze_list = [], train = 100):
    global chunks
    global labels
    global channeling
    global subbanding
    global iter_seconds
    global os_mav
    global directory_compare
    global directory_mav
    global directory_analysis
    global directory_encoded
    global directory_unique
    
    test = False

    for bw in subbanding:
        for channel in channeling:
            sub = channel + "_" + bw
            difference_list = {}
            for label in labels:
                filesave = label
                specific = label
                dif = ""
                dif_list = []
                encoded_list = {}
                compare_list = {}
                train_list = {}
                test_list = {}
                encoded_test_list = {}
                df_encoded_test_list = {}

                counter = 1
                filenames = []
                print(bw + " Files (" + label + ") " + channel + ":\n-------------------------")
                if not analyze_list:
                    for file in os.listdir(os_mav):
                        temp = []
    #                     mean_temp = []
    #                     mav_temp = []
    #                     sd_temp = []

                        filename = os.fsdecode(file)
                        sub = channel + "_" + bw
                        if filename.find(sub) != -1:
                            if filename.endswith(ext_raw):
                                if specific == "":
                                    print(str(counter) + ". " + filename)

                                    filepath = directory_mav + "/" + filename
                                    with open(filepath, newline='') as f:
                                        reader = csv.reader(f)
                                        data = list(reader)
                                        for x in data:
                                            temp.append(float(x[3]))
    #                                         mean_temp.append(float(x[1]))
    #                                         mav_temp.append(float(x[2]))
    #                                         sd_temp.append(float(x[3]))
                                        if len(temp) > 0:
                                            train_list[filename] = temp
                                            filenames.append(filename)
                                    counter += 1
                                else:
                                    if filename.find(specific) != -1:
                                        filepath = directory_mav + "/" + filename
                                        with open(filepath, newline='') as f:
                                            reader = csv.reader(f)
                                            data = list(reader)
                                            for x in data:
                                                temp.append(float(x[3]))
    #                                             mean_temp.append(float(x[1]))
    #                                             mav_temp.append(float(x[2]))
    #                                             sd_temp.append(float(x[3]))
                                            if len(temp) > 0:
                                                train_list[filename] = temp
                                                filenames.append(filename)
                    if train > 0 and train < 100:
                        test = True

                        train_keys = random.sample(list(train_list), int(train / 100 * len(train_list)))
                        filenames = []
                        temp_list = {}
                        for fn, al in train_list.items():
                            if fn in train_keys:
                                print(str(counter) + ". " + fn)
                                temp_list[fn] = al
                                filenames.append(fn)
                                counter += 1
                            else:
                                test_list[fn] = al
                        train_list = temp_list
                else:
                    if analyze_list[label]:
                        if train > 0 and train < 100:
                            test = True
                            train_temp = {}
                            test_temp = {}

                            train_keys = random.sample(list(analyze_list[label]), int(train / 100 * len(analyze_list[label])))
                            for fn, al in analyze_list[label].items():
                                if fn in train_keys:
                                    train_temp[fn] = al
                                else:
                                    test_temp[fn] = al
                            for fn, al in train_temp.items():
                                temp = []

                                for x in al[bw]:
                                    temp = x["SD"]

                                if len(temp) > 0:
                                    train_list[fn] = temp
                                    filenames.append(fn)

                                print(str(counter) + ". " + fn)

                                counter += 1
                            for fn, al in test_temp.items():
                                temp = []

                                for x in al[bw]:
                                    temp = x["SD"]

                                if len(temp) > 0:
                                    test_list[fn] = temp
                        else:
                            for fn, al in analyze_list[label].items():
                                temp = []
                                mean_temp = []
                                mav_temp = []
                                sd_temp = []

                                filename = fn
                                sub = channel + "_" + bw

                                for x in al[bw]:
                                    temp = x["SD"]

                                if len(temp) > 0:
                                    train_list[filename] = temp
                                    filenames.append(filename)

                                print(str(counter) + ". " + filename)

                                counter += 1
                    else:
                        print("\nNot found: " + label + "\n")

                print("-------------------------")

                if test:
                    for fn, tl in test_list.items():
                        encoded_test_list[fn] = encodeGrowthData(tl)
                    for key, data in encoded_test_list.items():
                        joined = "".join(map(str, data))
                        df_encoded_test_list[key] = joined

                if len(train_list) > 0:
                    for fn in filenames:
                        encoded_list[fn] = encodeGrowthData(train_list[fn])
                    for key1, data1 in encoded_list.items():
                        temp_list = {}

                        for key2, data2 in encoded_list.items():
                            if key1 != key2:
                                temp_list2 = []
                                result_list = compareGrowthEncoded(data1, data2)
                                for res in result_list:
                                    match = res[0]
                                    start = res[1]
                                    end = match + start
                                    matching = data1[start:end]
                                    match_string = "".join(str(v) for v in matching)
                                    temp_list2.append(match_string)
                                    dif_list.append(match_string)
                                temp_list[key2] = temp_list2
    #                             match_max, start1, start2 = compareGrowthEncoded(data1, data2)
    #                             temp_list.append({key2 : [match_max, start1, start2, data1, data2]})

                        compare_list[key1] = temp_list

                    fn = "compare-std-" + channel + "-" + bw + ".xlsx"
                    if filesave != "":
                        fn = "compare-std-" + channel + "-" + bw + "-" + filesave + ".xlsx"
                    final_list = {}
                    for key1, data1 in encoded_list.items():
                        for key2, data2 in compare_list[key1].items():
                            final_list[key1 + ' | ' + key2] = data2
                    dfFinal = pd.DataFrame.from_dict(final_list, orient='index')
                    dfFinal.to_excel(directory_compare + "/" + fn, index_label='Filenames')

                    df_encoded_list = {}
                    for key, data in encoded_list.items():
                        joined = "".join(map(str, data))
                        df_encoded_list[key] = joined
                    efn = directory_encoded + "/encoded_std_" + channel + "_" + bw + ".xlsx"
                    if filesave != "":
                        efn = directory_encoded + "/encoded_std_" + channel + "_" + bw + "_" + filesave + ".xlsx"
                    dfEncoded = pd.DataFrame.from_dict(df_encoded_list, orient='index', columns=['Encode'])
                    dfEncoded.to_excel(efn, index_label='Filename')

                    if df_encoded_test_list:
                        efn = directory_encoded + "/encoded_std_" + channel + "_" + bw + "_test.xlsx"
                        if filesave != "":
                            efn = directory_encoded + "/encoded_std_" + channel + "_" + bw + "_" + filesave + "_test.xlsx"
                        dfEncoded = pd.DataFrame.from_dict(df_encoded_test_list, orient='index', columns=['Encode'])
                        dfEncoded.to_excel(efn, index_label='Filename')

                    cfn = "compare_distance_std_" + channel + "_" + bw + "_"
                    if filesave != "":
                        cfn = cfn + filesave + "_"
    #                 lev_match = levMatch(dif_list, True, False, 2)
                    best_list = []
                    if test:
                        lev_min, min_lev, final_list, best_list = minLevMatch(dif_list, dif_list, True, 2, cfn + "full.xlsx", df_encoded_list, df_encoded_test_list)
                    else:
                        lev_min, min_lev, final_list, best_list = minLevMatch(dif_list, dif_list, True, 2, cfn + "full.xlsx", df_encoded_list)
    #                 lev_min, min_lev, final_list = minLevMatch(lev_match, dif_list, True, 5, cfn + "partial.xlsx")
    #                 lev_min, min_lev, final_list = minLevMatch(lev_match, lev_match, True, 5, cfn + "minimal.xlsx")
                    difference_list[label] = best_list
            
            print(difference_list)
            
            uniqueness = dict()
            for a, b in difference_list.items():
                temp = []
                banned = []
                for x, y in difference_list.items():
                    if a != x:
                        for element in b:
                            if element not in y:
                                if element not in temp:
                                    if element not in banned:
                                        temp.append(element)
                                else:
                                    temp.remove(element)
                                    if element not in banned:
                                        banned.append(element)
                uniqueness[a] = [', '.join(temp)]
                
            filename = directory_unique + "/unique_match_std_" + sub + ".xlsx"
            unique_list = pd.DataFrame.from_dict(uniqueness, orient='index', columns=['Encode List'])
            unique_list.to_excel(filename, index_label='Label')
                
    print()
    
def analyzePSD():
    global chunks
    global iter_seconds
    global os_processed
    global directory_psd
    global directory_compare
    global directory_analysis
    global directory_encoded
    global brainwaves

    dif = ""
    dif_list = []
    for bw in brainwaves:
        for channel in channels:
            if channel != 'Nan':
                data_list = dict()
                encoded_list = dict()
                compare_list = dict()

                counter = 1
                filenames = []
                print(bw + " Files " + channel + ":\n-------------------------")
                for file in os.listdir(os_processed):
                    temp = []
                    filename = os.fsdecode(file)
                    sub = channel + "_" + bw
                    if filename.find(sub) != -1:
                        if filename.endswith(".csv"):

                            print(str(counter) + ". " + filename)

                            filepath = directory_psd + "/" + filename
                            df = pd.read_csv(filepath)
                            with open(filepath, newline='') as f:
                                reader = csv.reader(f)
                                data = list(reader)
                                for x in data:
                                    for y in x:
                                        temp.append(float(y))
                                if len(temp) > 0:
                                    data_list[filename] = temp
                                    filenames.append(filename)
                            counter += 1
                print("-------------------------")

                if len(data_list) > 0:
                    for fn in filenames:
                        encoded_list[fn] = encodeGrowthData(data_list[fn])

                    for key1, data1 in encoded_list.items():
                        temp_list = []

                        for key2, data2 in encoded_list.items():
                            if key1 != key2:
                                temp_list2 = []
                                result_list = compareGrowthEncoded(data1, data2)
                                for res in result_list:
                                    match = res[0]
                                    start = res[1]
                                    end = match + start
                                    matching = data1[start:end]
                                    match_string = "".join(str(v) for v in matching)
                                    temp_list2.append(match_string)
                                    if bw == "Alpha" and channel == "T3":
                                        dif_list.append(match_string)
                                temp_list.append({key2 : temp_list2})
    #                             match_max, start1, start2 = compareGrowthEncoded(data1, data2)
    #                             temp_list.append({key2 : [match_max, start1, start2, data1, data2]})

                        compare_list[key1] = temp_list

                    fn = "compare-psd-" + channel + "-" + bw + ".xlsx"
                    final_list = []
                    for key1, data1 in encoded_list.items():
    #                     fn = "compare-psd-" + key1.replace(".csv","") + ".xlsx"
    #                     final_list = []

                        iter_check = int(math.floor(iter_seconds / chunks))
                        for urut in range(iter_check):
                            sem = {}
                            for key2, data2 in encoded_list.items():
                                if key1 != key2:
                                    for x in compare_list[key1]:
                                        for a, b in x.items():
                                            if a == key2:
                                                comparing = key1 + " | " + key2
                                                sem[comparing] = b[urut]
                            final_list.append(sem)
    #                     dfFinal = pd.DataFrame(final_list)
    #                     dfFinal = dfFinal.transpose()
    #                     dfFinal.to_excel(directory_compare + "/" + fn)
                    dfFinal = pd.DataFrame(final_list)
                    dfFinal = dfFinal.transpose()
                    dfFinal.to_excel(directory_compare + "/" + fn)

                    df_encoded_list = {}
                    for key, data in encoded_list.items():
                        joined = "".join(map(str, data))
                        df_encoded_list[key] = joined
                    efn = directory_encoded + "/encoded_psd_" + channel + "_" + bw + ".xlsx"
                    dfEncoded = pd.DataFrame.from_dict(df_encoded_list, orient='index')
                    dfEncoded.to_excel(efn)
                    
    lev_match = levMatch(dif_list, True, False, 2)
    lev_min, min_lev, final_list = minLevMatch(dif_list, dif_list, True, 2, "compare_distance_psd_full.xlsx")
    lev_min, min_lev, final_list = minLevMatch(lev_match, dif_list, True, 5, "compare_distance_psd_partial.xlsx")
    lev_min, min_lev, final_list = minLevMatch(lev_match, lev_match, True, 5, "compare_distance_psd_minimal.xlsx")
#     print("C1 Min = " + str(c1_min[0]) + " [" + str(c1_min[1]) + "]")
#     print("C2 Min = " + str(c2_min[0]) + " [" + str(c2_min[1]) + "]")
    print()
            
#             print()
#             for x, y in compare_list.items():
#                 print(x + ":")
#                 for z in y:
#                     for a, b in z.items():
#                         print()
#                         print(a)
#                         for c in b:
#                             print(c)
#                         match_max = b[0]
#                         start1 = b[1]
#                         start2 = b[2]
#                         end = start1 + match_max
#                         print()
#                         print(a + " = " + str(match_max) + " match")
#                         print("Index = [" + str(start1) + ", " + str(start2) + "]")
#                         print(b[3][start1:end])
                        
#                 print()

def levMatch(dif_list, print_iter, print_min, iter_mod = 2):
    global start_time
    start_time = time.time()
    
    lev_max = -1
    c1_max = []
    c2_max = []
    lev_min = -1
    c1_min = []
    c2_min = []
    same_match = []
    min_match = []
    
    dif_list_len = len(dif_list)
    iter_max = dif_list_len * (dif_list_len - 1)
    iter_counter = 1
    iter_temp = 0
    
    checked = []
    
    print()
    print("Iterations = " + str(iter_max))
    print()
    for c1, d1 in enumerate(dif_list):
        checked2 = []
        if d1 not in checked:
            checked.append(d1)
            for c2, d2 in enumerate(dif_list):
                if d2 not in checked2:
                    checked2.append(d2)
                    if print_iter:
                        iter_temp = printIter(iter_temp, iter_counter, iter_max, iter_mod)

                    if d1 != d2:
                            lev = 100
                            if len(d1) == len(d2):
                                lev = hamming(d1, d2)
                            else:
                                lev = levenshtein(d1, d2)
                            if lev > lev_max:
                                lev_max = lev
                                c1_max = [c1, d1]
                                c2_max = [c2, d2]
                            if lev < lev_min or lev_min == -1:
                                lev_min = lev
                                c1_min = [c1, d1]
                                c2_min = [c2, d2]
                            if lev == 1:
                                if d1 not in min_match:
                                    min_match.append(d1)
                                if d2 not in min_match:
                                    min_match.append(d2)
                    elif c1 != c2:
                        if d1 not in same_match:
                            same_match.append(d1)

                    iter_counter += 1
        
    same_max = 0
    max_same = []
    for sm in same_match:
        total = dif_list.count(sm)
        if total > same_max:
            same_max = total
            
    for sm in same_match:
        total = dif_list.count(sm)
        if total == same_max:
            max_same.append(sm)
    
    print("\nSame = " + str(len(same_match)) + "/" + str(iter_max) + " (Unique = " + str(same_max) + ")")
    print(max_same)
    print("\nLevenshtein Distance:")
    print("\nLev Max = " + str(lev_max))
    print("C1 Max = " + str(c1_max[0]) + " [" + str(c1_max[1]) + "]")
    print("C2 Max = " + str(c2_max[0]) + " [" + str(c2_max[1]) + "]")
    print("\nLev Min = " + str(lev_min))
    print("Min Match = " + str(len(min_match)))
    if print_min:
        for mm in min_match:
            print(mm) 
        
    return min_match

def minLevMatch(dif_list, ori_list, print_iter, iter_mod = 5, filename = "compare_distance.xlsx", encoded_list = [], test_list = {}):
    global start_time
    global directory_analysis
    global labels
    
    start_time = time.time()
    
    min_match = []
    
    iter_max = len(dif_list) * (len(ori_list))
    iter_counter = 1
    iter_temp = 0
    
    min_lev = ""
    lev_min = -1
    
    checked = []
    checked_list = {}
    
    analyze_list = dict()
    
    print()
    print("Iterations = " + str(iter_max))
    print()
    for c1, d1 in enumerate(dif_list):
        if len(d1) > 0:
    #         lev_list = []
            checked2 = []
            lev_total = 0
            lev_counter = 0

            if d1 not in checked:
                checked.append(d1)
                for c2, d2 in enumerate(ori_list):
                    if len(d2) > 0:
                        if d2 not in checked2:
                            checked2.append(d2)
                            if print_iter:
                                iter_temp = printIter(iter_temp, iter_counter, iter_max, iter_mod)

                            lev = 100
                            if d1 != d2:
                                if len(d1) == len(d2):
                                    lev = hamming(d1, d2)
                                else:
                                    lev = levenshtein(d1, d2)
        #                         lev_list.append(lev)
                            else:
                                lev = 0
                            checked_list[d2] = lev

                        lev_total += checked_list[d2]
                        lev_counter += 1
                        iter_counter += 1

    #             lev_mean = sum(lev_list) / len(lev_list)
                lev_mean = lev_total / lev_counter
                analyze_list[d1] = lev_mean
                if lev_min == -1 or lev_mean < lev_min:
                    lev_min = lev_mean
                    min_lev = d1
    
    final_list = dict()
    best_list = []
    columns = ['Distance']
    if encoded_list:
        columns = ['Length','Distance 1','Distance 2','Existences','% Occurrences','AVG Occurrences']
        if test_list:
            columns = ['Length','Distance 1','Distance 2','Existences 1','% Occurrences 1','AVG Occurrences 1','Existences 2','% Occurrences 2','AVG Occurrences 2']
        for a, b in analyze_list.items():
            exist = 0
            occurrence = 0
            pieces = 0
            lev = 0
            total = len(encoded_list)
            for x, y in encoded_list.items():
                length_match = len(a)
                length_encode = len(y)
                length_max = length_encode / length_match
                occur = y.count(a)
                pieces += occur / length_max
                occurrence += occur
                if occur > 0:
                    exist += 1
                if len(a) == len(y):
                    lev += hamming(a, y)
                else:
                    lev += levenshtein(a, y)
            existences = ("%.2f%%" % (100 * exist / total))
            average = ("%.2f%%" % (100 * pieces / total))
            occurrences = math.floor((occurrence / total) * 100) / 100
            distance = math.floor((lev / total) * 100) / 100
            if test_list:
                existences_test = '-'
                average_test = '-'
                occurrences_test = 0
                if existences == "100.00%":
                    exist_test = 0
                    occurrence_test = 0
                    pieces_test = 0
                    total_test = len(test_list)
                    for x, y in test_list.items():
                        length_match = len(a)
                        length_encode = len(y)
                        length_max = length_encode / length_match
                        occur_test = y.count(a)
                        pieces_test += occur_test / length_max
                        occurrence_test += occur_test
                        if occur_test > 0:
                            exist_test += 1
                    existences_test = ("%.2f%%" % (100 * exist_test / total_test))
                    average_test = ("%.2f%%" % (100 * pieces_test / total_test))
                    occurrences_test = math.floor((occurrence_test / total_test) * 100) / 100
                    if existences_test == "100.00%":
                        best_list.append(a)
                final_list[a] = [len(a), b, distance, existences, average, occurrences, existences_test, average_test, occurrences_test]
            else:
                final_list[a] = [len(a), b, distance, existences, average, occurrences]
    else:
        final_list = analyze_list
        
    print("\nLev Mean Min = " + str(lev_min) + " (" + min_lev + ")")
    
    filename = directory_analysis + "/" + filename
    distance_list = pd.DataFrame.from_dict(final_list, orient='index', columns=columns)
    distance_list.to_excel(filename, index_label='Encode')
            
    return lev_min, min_lev, final_list, best_list

def printIter(iter_temp, iter_counter, iter_max, per = 5):
    global start_time
    global end_time
    
    mod = 2
    if iter_max > 1000000:
        mod = 1000
    elif iter_max > 10000:
        mod = 100
    elif iter_max > 1000:
        mod = 10
    iter_mod = iter_counter % mod
    
    if iter_mod == 0:
        iter_per = iter_counter / iter_max * 100
        if int(iter_per) > iter_temp:
            if int(iter_per) % per == 0:
                end_time = time.time()
                print(str(int(iter_per)) + "% (" + printTimer(int(end_time - start_time)) + ")")
                iter_temp = int(iter_per)
                
    return iter_temp
                        
def compareGrowthEncoded(data_list1, data_list2):
    global chunks
    global iter_seconds
    
    iter_max = len(data_list1)
    if len(data_list2) < iter_max:
        iter_max = len(data_list2)
    iter_check = int(math.floor(iter_seconds / chunks))
    
    data_list = []
    result_list = []
    result = 0
    start1 = -1
    start2 = -1
    for i in range(iter_check):
        winner = 0
        match = 0
        temp1 = -1
        temp2 = -1
        before = False
        for n in range(iter_max):
            cur = n+i
            
            if cur < iter_max:
                if data_list1[cur] == data_list2[n]:
                    if match == 0 or before:
                        match += 1
                        before = True
                        if match > winner:
                            winner = match
                            temp1 = cur
                            temp2 = n
                    else:
                        before = False
                        if match > winner:
                            winner = match
                            temp1 = cur
                            temp2 = n
                else:
                    before = False
                    if match > winner:
                        winner = match
                        temp1 = cur
                        temp2 = n
                    match = 0
                        
            if winner > result:
                result = winner
                start1 = temp1
                start2 = temp2
                    
        result_list.append([winner, start1+1-winner])
            
    return result_list
#     return result, start1+1-result, start2+1-result
                        
def encodeGrowthData(data_list):
    encoded_list = []
    first = 0
    second = 0
    check = False
    for data in data_list:
        second = data
        if check:
            if first < second:
                encoded_list.append(1)
            else:
                encoded_list.append(0)
        else:
            check = True
        first = data
        
    return encoded_list

def hamming(s1,s2):
    result = -1
    if len(s1) != len(s2):
        return result
    else:
        for x, (i, j) in enumerate(zip(s1, s2)):
            if i != j:
                result += 1
    return result

def levenshtein(seq1, seq2, print_matrix = False):
    size_x = len(seq1) + 1
    size_y = len(seq2) + 1
    matrix = np.zeros ((size_x, size_y))
    for x in range(size_x):
        matrix [x, 0] = x
    for y in range(size_y):
        matrix [0, y] = y

    for x in range(1, size_x):
        for y in range(1, size_y):
            if seq1[x-1] == seq2[y-1]:
                matrix [x,y] = min(
                    matrix[x-1, y] + 1,
                    matrix[x-1, y-1],
                    matrix[x, y-1] + 1
                )
            else:
                matrix [x,y] = min(
                    matrix[x-1,y] + 1,
                    matrix[x-1,y-1] + 1,
                    matrix[x,y-1] + 1
                )
    if print_matrix:
        print (matrix)
    return (matrix[size_x - 1, size_y - 1])

def calculateFeature(subbandData):
#     print("\nSubbandData[0]")
#     print(subbandData[0])
#     print("\n\n")
    mean = statistics.mean(subbandData[0])
    var = statistics.variance(subbandData[0])
    sd = var ** (0.5)
    absolute = [abs(ele) for ele in subbandData[0]]
    mav = statistics.mean(absolute)
    value, counts = np.unique(subbandData[0], return_counts=True)
    en = entropy(counts, base=None)
    mob, com = hjorth_params(subbandData[0])
    
#     print("\nEntropy = " + str(en) + "\n")
    
#     return float(mean), float(np.around(mav,5)), float(variance), float(sd), float(en)
#     return float(mean), float(np.around(mav,5)), float(var), float(sd), float(en), float(mob), float(com)
    return float(mean), float(mav), float(var), float(sd), float(en), float(mob), float(com)
        
def comparePSD(wave):
    global os_analysis
    
    bw = ""
    if wave == 1:
        bw = "Delta"
    elif wave == 2:
        bw = "Theta"
    elif wave == 3:
        bw = "Alpha"
    elif wave == 4:
        bw = "Beta"
    elif wave == 5:
        bw = "Gamma"
    else:
        print("\nInput doesn't work, please see input list!")
    
    if bw != "":
        for channel in channels:
            if channel != 'Nan':
                data_list = []
                counter = 1
                filenames = []
                print("Files " + channel + ":\n-------------------------")
                for file in os.listdir(os_analysis):
                    temp = []
                    filename = os.fsdecode(file)
                    sub = channel + "_" + bw 
                    if filename.find(sub) != -1:
                        if filename.endswith(".csv"):

                            print(str(counter) + ". " + filename)

                            filepath = directory_psd + "/" + filename
                            df = pd.read_csv(filepath)
                            with open(filepath, newline='') as f:
                                reader = csv.reader(f)
                                data = list(reader)
                                for x in data:
                                    for y in x:
                                        temp.append(float(y))
                                if len(temp) > 0:
                                    data_list.append(temp)
                                    filenames.append(filename)
                            counter += 1
                print("-------------------------")

                if len(data_list) > 0:
                    dfPSD = pd.DataFrame(data_list)
                    dfPSD = dfPSD.transpose()
                    dfPSD.columns = filenames
                    dfMean = dfPSD[filenames].mean()
                    print("\nAverage:")
                    print(dfMean)
                    dfPSD.plot(kind='line',figsize=(15,6))
                    plt.style.use('seaborn-colorblind')
                    plt.style.use('seaborn-whitegrid')
                    plt.title("PSD Analysis " + sub)
                    plt.xlabel("Frame")
                    plt.ylabel("Value")
                    plt.legend()
                    graph_title = directory_graph + "/Analysis_PSD_"+ sub + ".png"
                    fileOverwrite(graph_title)
                    plt.savefig(graph_title)
                    plt.show()

def fileOverwrite(filepath):
    if os.path.exists(filepath):
        os.remove(filepath)

def printCurrentSettings():
    global timer
    global channels
    global second_start
    global seconds
    global chunks
    
    print("\nCurrent channels:")
    print(*channels, sep = ", ")
    print("\nLive Graph EEG:")
    print("Current measurement time = " + str(measurement_time))
    print("\nLive EEG:")
    print("Current timer (in seconds) = " + str(timer))
    print("\nFolder EEG:")
    print("Current starting second = " + str(second_start))
    print("Current seconds = " + str(seconds))
    print("\nAll EEG:")
    print("Current chunk seconds = " + str(chunks))
    
def printTimer(seconds, before = ""):
    minutes = 60
    hours = minutes * 60
    days = hours * 24
    
    result = ""
    left = 0
    if seconds >= days:
        left = seconds % days
        day = int(math.floor(seconds / days))
        if before != "":
            before = before + " "
        if day > 0:
            before = before + str(day) + " day"
        if day > 1:
            before = before + "s"
        result = printTimer(left, before)
    elif seconds >= hours:
        left = seconds % hours
        hour = int(math.floor(seconds / hours))
        if before != "":
            before = before + " "
        if hour > 0:
            before = before + str(hour) + " hour"
        if hour > 1:
            before = before + "s"
        result = printTimer(left, before)
    elif seconds >= minutes:
        left = seconds % minutes
        minute = int(math.floor(seconds / minutes))
        if before != "":
            before = before + " "
        if minute > 0:
            before = before + str(minute) + " minute"
        if minute > 1:
            before = before + "s"
        result = printTimer(left, before)
    else:
        if before != "" and seconds > 0:
            before = before + " "
        if seconds > 0:
            before = before + str(seconds) + " second"
        if seconds > 1:
            before = before + "s"
        result = before
    
    return result

def copyFile(source_folder, destination_folder, filename):
    source_path = os.path.join(source_folder, filename)
    destination_path = os.path.join(destination_folder, filename)
    fileOverwrite(destination_path)
    
    shutil.copy(source_path, destination_path)

In [4]:
def main():
    global timer
    global channels
    global brainwaves
    global second_start
    global seconds
    global cut
    global dict_zip
    global frame_start
    global frame_end
    global chunks
    global sample_rate
    global measurement_time
    global do_analyze_psd
    global do_analyze_encode
    global interval_sec
    global final_graph
    global live_filename
    global start_time
    global end_time
    global train_encode
    global labels
    global label_prefix
    global label_prefix_start
    
    global debug
    
    global os_raw
    global os_analysis
    global os_processed
    
    timer = int(timer)
    filerename = "temp"
    
    print("Program Start!")
    
#     printCurrentSettings(timer, channels, second_start, seconds)
    printCurrentSettings()
    
    running = True
    choice = 0
    
    while running:
        ica_list = []
        delta_list = []
        theta_list = []
        alpha_list = []
        beta_list = []
        gamma_list = []
        
        print("\n[0] Enter 0 to run live graph EEG.")
        print("[1] Enter 1 to run live EEG.")
        print("[2] Enter 2 to run folder EEG.")
        print("[3] Enter 3 to run PSD analysis.")
        print("[4] Enter 4 to change channels.")
        print("[5] Enter 5 to change timer (Live EEG).")
        print("[6] Enter 6 to change starting second (Folder EEG).")
        print("[7] Enter 7 to change seconds (Folder EEG).")
        print("[8] Enter 8 to change chunks (All EEG).")
        print("[9] Enter 9 to change measurement time (Live Graph EEG).")
        print("[A] Enter A to run compare PSD growth.")
        print("[B] Enter B to run compare MAV growth.")
        print("[C] Enter C to run train test encode analyze.")
        print("[D] Enter D to run analyze PSD.")
        print("[L] Enter L to change live EEG filename.")
        print("[s] Enter s to check current settings.")
        print("[q] Enter q to quit.")
        
        choice = input("Enter input: ")
        
        if choice == '0':
            filename = "Live EEG.txt"
            frame_start = 1
            frame_end = (timer * sample_rate) - 1
            if chunks > timer:
                chunks = timer
                if chunks > 1:
                    chunks -= 1
                
            print("\nRunning live graph EEG\n")
           
            plt.style.use('seaborn-colorblind')
            plt.style.use('seaborn-whitegrid')
            
            live_data = []
        
            final_graph = False
        
            %matplotlib qt    
            for i in range(measurement_time):
                dfRaw = getLiveData()
#                 plotRawData(dfRaw, filename)
                print("Step " + str(i))
                outputSignals = filterRawData(dfRaw)
                butters = butterData(outputSignals)
                icas = ICAData(butters)
#                 plotAmp(icas, filename)
                saveICAcsv(icas, filename)

                for idx, icad in enumerate(icas):
                    ica_list.append(icad)

                delta, theta, alpha, beta, gamma = psdData(icas, filename)
                
                # meanMutli to plot all
                
                for d in delta:
                    live_data.append(d)
                
                cur_channel = 0    
                for ld in live_data:
                    label = channels[cur_channel] + " Delta"
                    plt.plot(ld, label = label)
                    cur_channel += 1
                
                plt.legend()

                plt.ylim(-0.2, 1.2)
                plt.title(f'FRAME {i+1}')

                if (i != measurement_time-1):
                    print("next\n")
                    plt.draw()
                    plt.pause(interval_sec)
                    plt.cla()
                else:
                    print("end")
                    plt.show()
            
            final_graph = True
                    
        elif choice == '1':
#             filename = live_filename + ".txt"
#             frame_start = 1
#             frame_end = (timer * sample_rate) - 1
#             if chunks > timer:
#                 chunks = timer
#                 if chunks > 1:
#                     chunks -= 1
            
            print("\nRunning live EEG (" + live_filename + ")\n")
            
            dfRaw = getLiveData()
            
        elif choice == '2':
            total_time = 0
            start_time_2 = time.time()
            if not os.path.exists(directory_ica):
                os.makedirs(directory_ica)
            if not os.path.exists(directory_psd):
                os.makedirs(directory_psd)
            if not os.path.exists(directory_graph):
                os.makedirs(directory_graph)
            if not os.path.exists(directory_analysis):
                os.makedirs(directory_analysis)
            if not os.path.exists(directory_mav):
                os.makedirs(directory_mav)
            if not os.path.exists(directory_compare):
                os.makedirs(directory_compare)
            if not os.path.exists(directory_encoded):
                os.makedirs(directory_encoded)
            if not os.path.exists(directory_unique):
                os.makedirs(directory_unique)
            if not os.path.exists(directory_corrupt):
                os.makedirs(directory_corrupt)
            if not os.path.exists(directory_empty):
                os.makedirs(directory_empty)
            
            print("\nRunning folder EEG\n")
            
            label_list = {}
            for label, data in labels.items():
                label_list[label] = {}
            
            raw_channels = []
            if ext_raw == '.xlsx':
                raw_channels.append('Timestamp')
                for x in range(len(channels)):
                    raw_channels.append('CH.' + str(x))
            else:
                raw_channels = channels
            
            counter = 0
            
            temp = ""
            empty_list = []
            corrupt_list = []
            for label, data in labels.items():
                finder = label_prefix + label
                if label_prefix_start == 1:
                    finder = label + label_prefix
                    
                second_start = data['start']
                seconds = data['seconds']

                print("-------------------------")
                print(finder)
                print("-------------------------")

                end_time = time.time()
                count_time = int(end_time - start_time)
                total_time = int(end_time - start_time_2)
                print("\nTime = (" + printTimer(count_time) + ")")
                print("Total Time = (" + printTimer(total_time) + ")\n")
                    
                for file in os.listdir(os_raw):
                    filename = os.fsdecode(file)

                    if filename.endswith(ext_raw):
                        if filename.find(finder) != -1:

                            print("-------------------------")
                            print(filename)
                            print("-------------------------")
                            
                            start_time = time.time()
                            filepath = directory_raw + "/" + filename
                            dfRaw = readRawData(filepath)
                            
                            if ext_raw == '.xlsx':
                                sample_rate = getSampleRate(dfRaw['Timestamp'])
                            else:
                                sample_rate = 256
                            print("\nSample Rate = " + str(sample_rate) + "\n")
                            if sample_rate is None:
                                empty_list.append(filename)
                                continue
                            temp = label
                            
                            frame_start, frame_end = getFrame(dfRaw)
            
#                             frame_start = second_start * sample_rate
#                             frame_end = ((second_start + seconds) * sample_rate) - 1
                        
                            dfClean = dfRaw[raw_channels].loc[frame_start:frame_end]

                            print("Finished Read Raw Data " + filename)
                            end_time = time.time()
                            count_time = int(end_time - start_time)
                            total_time = int(end_time - start_time_2)
                            print("\nTime = (" + printTimer(count_time) + ")")
                            print("Total Time = (" + printTimer(total_time) + ")\n")
                            
                            dfClean.sort_values('Timestamp', ascending=True, inplace=True)
                            
                            dfTimestamp = dfClean['Timestamp']

                            try:
    #                             plotRawData(dfRaw, filename)
                                outputSignals = filterRawData(dfClean)
    #                             plotFilterData(outputSignals, filename)
                                butters = butterData(outputSignals)
    #                             plotButterData(butters, filename)
                                icas = ICAData(butters)
    #                             plotAmp(icas, filename, False)
    #                             saveICAcsv(icas, filename)

                                for idx, icad in enumerate(icas):
                                    ica_list.append(icad)

                                if do_psd:
                                    delta, theta, alpha, beta, gamma = psdData(icas, filename, False, dfTimestamp)
                                mav_delta, mav_theta, mav_alpha, mav_beta, mav_gamma = mavData(icas, filename, False, dfTimestamp)

    #                             delta_list.append(delta)
    #                             theta_list.append(theta)
    #                             alpha_list.append(alpha)
    #                             beta_list.append(beta)
    #                             gamma_list.append(gamma)

                                testing_list = {
                                    "Delta" : mav_delta,
                                    "Theta" : mav_theta,
                                    "Alpha" : mav_alpha,
                                    "Beta" : mav_beta,
                                    "Gamma" : mav_gamma
                                }

                                label_list[label][filename] = testing_list
                            except ValueError:
                                corrupt_list.append(filename)
                                continue

                            print("Finished Preprocessed " + filename)
                            end_time = time.time()
                            count_time = int(end_time - start_time)
                            total_time = int(end_time - start_time_2)
                            print("\nTime = (" + printTimer(count_time) + ")")
                            print("Total Time = (" + printTimer(total_time) + ")\n")

                            counter += 1
                            
                            if debug:
                                break
                    if debug:
                        break
                #end of for each file
            
#             if do_analyze_psd:
#                 print("\nRunning analyze PSD\n")
            
#                 for label, data in labels.items():
#                     finder = label_prefix + label
#                     if label_prefix_start == 1:
#                         finder = label + label_prefix

#                     second_start = data['start']
#                     seconds = data['seconds']

#                     print("-------------------------")
#                     print(finder)
#                     print("-------------------------")

#                     end_time = time.time()
#                     count_time = int(end_time - start_time)
#                     total_time = int(end_time - start_time_2)
#                     print("\nTime = (" + printTimer(count_time) + ")")
#                     print("Total Time = (" + printTimer(total_time) + ")\n")

#                     for file in os.listdir(os_processed):
#                         filename = os.fsdecode(file)

#                         if filename.endswith(".csv"):
#                             if filename.find(finder) != -1:

#                                 print("-------------------------")
#                                 print(filename)
#                                 print("-------------------------")
                                
#                                 filepath = directory_psd + "/" + filename
                                
#                                 dfRaw = pd.read_excel(filepath)
                                
                
            if do_analyze_encode:
                analyzeStatistic(label_list, train_encode)

    #             print("\nTotal EEG = " + str(counter))
    #             print()
    #             print("Mean Delta:")
    #             meanMulti(delta_list)
    #             print()
    #             print("Mean Theta:")
    #             meanMulti(theta_list)
    #             print()
    #             print("Mean Alpha:")
    #             meanMulti(alpha_list)
    #             print()
    #             print("Mean Beta:")
    #             meanMulti(beta_list)
    #             print()
    #             print("Mean Gamma:")
    #             meanMulti(gamma_list)
    #             print()

            end_time_2 = time.time()
            print("\n\nFINISH = (" + printTimer(int(end_time_2 - start_time_2)) + ")\n\n")
            
            source_path = directory_raw
            destination_path = directory_empty
            for empty in empty_list:
                copyFile(source_path, destination_path, empty)
                
            source_path = directory_raw
            destination_path = directory_corrupt
            for corrupt in corrupt_list:
                copyFile(source_path, destination_path, corrupt)
            
            empty_string = '\n'.join(empty_list)
            corrupt_string = '\n'.join(corrupt_list)
            with open('EEG Failed Dataset.txt', 'w') as file:
                file.write("Empty:\n")
                file.write(empty_string)
                file.write("\n\nCorrupt:\n")
                file.write(corrupt_string)
            
        elif choice == '3':
            print()
            print("[1] Enter 1 for Delta.")
            print("[2] Enter 2 for Theta.")
            print("[3] Enter 3 for Alpha.")
            print("[4] Enter 4 for Beta.")
            print("[5] Enter 5 for Gamma.")
            wave = int(input("\nInput brainwave code (above) to analyze: "))
            print()
            comparePSD(wave)
            
        elif choice == '4':
            channels = []
            
            n = int(input("\nEnter number of channels: "))
            
            for i in range(0, n):
                x = i + 1
                txt = "Channel " + str(x) + " = "
                channel = input(txt)
                channels.append(channel)
                
            cut = list(range(1, len(channels)+1))
            dict_zip = dict(zip(cut, channels))
                
            print("\nNew channels:")
            print(*channels, sep = ", ")
            
        elif choice == '5':
            timer = int(input("\nInput new timer (in seconds): "))
            
        elif choice == '6':
            second_start = int(input("\nInput new starting second: "))
            
        elif choice == '7':
            seconds = int(input("\nInput new seconds: "))
            
        elif choice == '8':
            chunks = int(input("\nInput new chunk seconds: "))
            
        elif choice == '9':
            measurement_time = int(input("\nInput new live graph measurement time: "))
            
        elif choice == 'A':
            print()
            analyzePSD()
            
        elif choice == 'B':
            print()
            analyzeStatistic()
            
        elif choice == 'C':
            print()
            analyzeStatistic([], "senang", 50, "-senang")
            analyzeStatistic([], "sedih", 50, "-sedih")
#             analyzeStatistic([] , "baseline", 50, "-baseline")

        elif choice == 'D':
            total_time = 0
            start_time_2 = time.time()
            
            print("\nRunning analyze PSD\n")
            
            label_list = {}
            for label, data in labels.items():
                label_list[label] = {}
            
            raw_channels = channels
            
            counter = 0
            
            temp = ""
            empty_list = []
            corrupt_list = []
            
            data_frames = []

            for label, data in labels.items():
                finder = label_prefix + label
                if label_prefix_start == 1:
                    finder = label + label_prefix

                second_start = data['start']
                seconds = data['seconds']

                print("-------------------------")
                print(finder)
                print("-------------------------")

                end_time = time.time()
                count_time = int(end_time - start_time)
                total_time = int(end_time - start_time_2)
                print("\nTime = (" + printTimer(count_time) + ")")
                print("Total Time = (" + printTimer(total_time) + ")\n")

                column_temp = {}
                for x in channels:
                    for y in brainwaves:
                        cn = x + '_' + y;
                        column_temp[cn] = []
                row_temp = {}
                
                initiate = True
                
                for file in os.listdir(os_processed):
                    filename = os.fsdecode(file)

                    if filename.endswith(ext_raw):
                        if filename.find(finder) != -1:

                            print("-------------------------")
                            print(filename)
                            print("-------------------------")

                            filepath = directory_psd + "/" + filename

                            dfPSD = pd.read_excel(filepath)
                            
#                             data_frames.append(dfPSD)
                            
                            column_first = dfPSD.columns[0]

                            if initiate:
                                for row_index, row in dfPSD.iterrows():
                                    for cn in column_temp:
                                        row_temp[row[column_first]] = {cn: [] for cn in column_temp}
                                initiate = False
                            
                            for idx, row in dfPSD.iterrows():
                                for cn in column_temp:
                                    row_temp[row[column_first]][cn].append(row[cn])
                
#                 print()
#                 print(len(column_temp['F8_Delta']))
                for cn in column_temp:
                    for idx, row in row_temp.items():
                        column_temp[cn].append(sum(row[cn]) / len(row[cn]))
#                 print()
#                 print([sum(row_temp[1.0]['F8_Delta']), len(row_temp[1.0]['F8_Delta'])])
#                 print()
#                 print(len(column_temp['F8_Delta']))
                
                filename = directory_analysis + "/psd_" + label + ".xlsx"
                dfAnalyze = pd.DataFrame.from_dict(column_temp)
                dfAnalyze.index = dfAnalyze.index + 1
                
                column_means = dfAnalyze.mean()

                mean_row = pd.DataFrame([column_means], columns=dfAnalyze.columns)
                mean_row = mean_row.rename(index={0: 'Average'})

                dfMean = pd.concat([dfAnalyze, mean_row])
                
                dfMean.to_excel(filename, index_label="Chunk") 
                
#                 combined_data = pd.concat(data_frames)
#                 combined_data.to_excel("output.xlsx") 
#                 print('combined_data:')
#                 print(combined_data)
#                 break
            
        elif choice == 'L':
            live_filename = str(input("\nInput new live EEG filename: "))
            
        elif choice == 's':
            printCurrentSettings()
            
        elif choice == 'q':
            running = False
        else:
            print("\nInput doesn't work, please see input list!")
            
        choice = 0
        
    print("\nProgram End!")

In [5]:
if __name__ == "__main__":
    main()

Program Start!

Current channels:
F8, F7, T8, T7, P8, P7, O2, O1

Live Graph EEG:
Current measurement time = 5

Live EEG:
Current timer (in seconds) = 5

Folder EEG:
Current starting second = 0
Current seconds = 5

All EEG:
Current chunk seconds = 0.25

[0] Enter 0 to run live graph EEG.
[1] Enter 1 to run live EEG.
[2] Enter 2 to run folder EEG.
[3] Enter 3 to run PSD analysis.
[4] Enter 4 to change channels.
[5] Enter 5 to change timer (Live EEG).
[6] Enter 6 to change starting second (Folder EEG).
[7] Enter 7 to change seconds (Folder EEG).
[8] Enter 8 to change chunks (All EEG).
[9] Enter 9 to change measurement time (Live Graph EEG).
[A] Enter A to run compare PSD growth.
[B] Enter B to run compare MAV growth.
[C] Enter C to run train test encode analyze.
[D] Enter D to run analyze PSD.
[L] Enter L to change live EEG filename.
[s] Enter s to check current settings.
[q] Enter q to quit.
Enter input: D

Running analyze PSD

-------------------------
baseline1_
----------------------

-------------------------
bengbeng_dataresponden8.xlsx
-------------------------
-------------------------
bengbeng_dataresponden9.xlsx
-------------------------
-------------------------
marjan_
-------------------------

Time = (42 seconds)
Total Time = (37 seconds)

-------------------------
marjan_dataresponden1.xlsx
-------------------------
-------------------------
marjan_dataresponden10.xlsx
-------------------------
-------------------------
marjan_dataresponden11.xlsx
-------------------------
-------------------------
marjan_dataresponden12.xlsx
-------------------------
-------------------------
marjan_dataresponden13.xlsx
-------------------------
-------------------------
marjan_dataresponden14.xlsx
-------------------------
-------------------------
marjan_dataresponden15.xlsx
-------------------------
-------------------------
marjan_dataresponden16.xlsx
-------------------------
-------------------------
marjan_dataresponden17.xlsx
-------------------------
-----------

-------------------------
pucuk_dataresponden14.xlsx
-------------------------
-------------------------
pucuk_dataresponden15.xlsx
-------------------------
-------------------------
pucuk_dataresponden16.xlsx
-------------------------
-------------------------
pucuk_dataresponden17.xlsx
-------------------------
-------------------------
pucuk_dataresponden18.xlsx
-------------------------
-------------------------
pucuk_dataresponden19.xlsx
-------------------------
-------------------------
pucuk_dataresponden2.xlsx
-------------------------
-------------------------
pucuk_dataresponden20.xlsx
-------------------------
-------------------------
pucuk_dataresponden22.xlsx
-------------------------
-------------------------
pucuk_dataresponden23.xlsx
-------------------------
-------------------------
pucuk_dataresponden24.xlsx
-------------------------
-------------------------
pucuk_dataresponden25.xlsx
-------------------------
-------------------------
pucuk_dataresponden26.xlsx


-------------------------
ts_nana_dataresponden30.xlsx
-------------------------
-------------------------
ts_nana_dataresponden4.xlsx
-------------------------
-------------------------
ts_nana_dataresponden5.xlsx
-------------------------
-------------------------
ts_nana_dataresponden6.xlsx
-------------------------
-------------------------
ts_nana_dataresponden7.xlsx
-------------------------
-------------------------
ts_nana_dataresponden8.xlsx
-------------------------
-------------------------
ts_nana_dataresponden9.xlsx
-------------------------
-------------------------
ts_pat_
-------------------------

Time = (1 minute 26 seconds)
Total Time = (1 minute 20 seconds)

-------------------------
ts_pat_dataresponden1.xlsx
-------------------------
-------------------------
ts_pat_dataresponden10.xlsx
-------------------------
-------------------------
ts_pat_dataresponden11.xlsx
-------------------------
-------------------------
ts_pat_dataresponden12.xlsx
--------------------

-------------------------
bearbrand_dataresponden15.xlsx
-------------------------
-------------------------
bearbrand_dataresponden16.xlsx
-------------------------
-------------------------
bearbrand_dataresponden17.xlsx
-------------------------
-------------------------
bearbrand_dataresponden18.xlsx
-------------------------
-------------------------
bearbrand_dataresponden19.xlsx
-------------------------
-------------------------
bearbrand_dataresponden2.xlsx
-------------------------
-------------------------
bearbrand_dataresponden20.xlsx
-------------------------
-------------------------
bearbrand_dataresponden22.xlsx
-------------------------
-------------------------
bearbrand_dataresponden23.xlsx
-------------------------
-------------------------
bearbrand_dataresponden24.xlsx
-------------------------
-------------------------
bearbrand_dataresponden25.xlsx
-------------------------
-------------------------
bearbrand_dataresponden26.xlsx
-------------------------
-----

KeyError: 360.0

In [ ]:
# directory_raw = "eye_tracker_raw"

# ext_raw = ".xlsx"

# os_raw = os.fsencode(directory_raw)

# for file in os.listdir(os_raw): filename = os.fsdecode(file)

# if filename.endswith(ext_raw):

#     print("-------------------------")
#     print(filename)
#     print("-------------------------")

#     start_time = time.time()
#     filepath = directory_raw + "/" + filename
#     dfRaw = pd.read_excel(filepath)
    
#     dfClean = dfRaw[(dfRaw['ET_ValidityLeft'] == 0) & (dfRaw['ET_ValidityRight'] == 0)]
#     dfClean.reset_index(drop=True, inplace=True)
    
#     print(dfClean)